# Finding Fine Tuning Params - lr and epochs #

In [2]:
from pathlib import Path
import time
import json
from tqdm.auto import tqdm
import gc

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
import sys
scripts_dir = (Path.cwd() / ".." / "scripts").resolve()
sys.path.append(str(scripts_dir))

from config import final_data_path, hf_token


/opt/anaconda3/envs/thesis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!pip install -U tabpfn

Setting variables for data paths.

In [ ]:
final_data_path = Path(final_data_path)
features_dir = final_data_path
results_dir = final_data_path / "modeling_results" / "fm_tuned"
results_dir.mkdir(parents=True, exist_ok=True)

layer_files = [
    "layer_a_bts",
    "layer_b_bts_meteostat",
    "layer_c_bts_gdelt",
    "layer_d_bts_meteostat_gdelt",
]

target_col = "dep_del15"
non_feature_cols = ["flight_id", "year", target_col]

datasets = {}
for name in layer_files:
    key = f"{name}_subsampled"
    df = pd.read_parquet(features_dir / f"{key}_features.parquet")
    datasets[key] = df
    print(f"{key}: {df.shape}")

layer_a_bts_subsampled: (6966559, 28)
layer_b_bts_meteostat_subsampled: (6966559, 32)
layer_c_bts_gdelt_subsampled: (6966559, 43)
layer_d_bts_meteostat_gdelt_subsampled: (6966559, 47)


In [5]:
def get_train_test(key: str):
    """Split one dataset by year and drop non-feature columns from X."""
    df = datasets[key]
    train = df[df["year"] == 2024]
    test = df[df["year"] == 2025]

    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X_train, y_train = train[feature_cols], train[target_col]
    X_test, y_test = test[feature_cols], test[target_col]
    return X_train, y_train, X_test, y_test

In [6]:
import os
os.environ["TABPFN_TOKEN"] = hf_token

In [7]:
from tabpfn.finetuning import FinetunedTabPFNClassifier

Creating a 100 000 row subsample for the parameter search using Layer D.

In [9]:
search_sample_size = 100_000
ft_search_layers = ["layer_d_bts_meteostat_gdelt_subsampled"]

ft_search_samples = {}
for search_layer in ft_search_layers:
    X_train_full, y_train_full, _, _ = get_train_test(search_layer)
    ft_search_df = X_train_full.copy()
    ft_search_df[target_col] = y_train_full.values
    ft_search_sample = ft_search_df.groupby(target_col).sample(
    frac=search_sample_size / len(ft_search_df), random_state=42
)

    X_search = ft_search_sample.drop(columns=[target_col])
    y_search = ft_search_sample[target_col]
    X_ft_train, X_ft_val, y_ft_train, y_ft_val = train_test_split(
        X_search, y_search, test_size=0.15, stratify=y_search, random_state=42
    )
    ft_search_samples[search_layer] = (X_ft_train, X_ft_val, y_ft_train, y_ft_val)
    print(f"{search_layer}: {len(X_search):,} rows sampled")

layer_d_bts_meteostat_gdelt_subsampled: 100,000 rows sampled


Running the search for the best learning rate and number of epochs for finetuning.

In [ ]:
class HistoryLogger:
    """Minimal logger conforming to tabpfn.finetuning.logging.FinetuningLogger --
    captures per-epoch metrics and prints live, so progress is visible during the run
    instead of dumping all at once when fit() returns."""
    def __init__(self):
        self.epoch_records = []

    def setup(self, config): pass
    def log_step(self, metrics, step): pass
    def log_epoch(self, metrics, step):
        self.epoch_records.append({**metrics, "step": step})
        if "val/ROC AUC" in metrics:
            print(f"    epoch {int(metrics['train/epoch'])}: val_auc={metrics['val/ROC AUC']:.4f}", flush=True)
    def finish(self): pass


tabpfn_n_estimators = 2
tabpfn_subsample_samples = 20_000
learning_rates = [3e-5, 1e-5, 3e-6, 1e-6, 3e-7]
max_epochs = 30
patience = 8
val_metric_key = "val/ROC AUC"

ft_search_results = []
ft_checkpoint_dir = results_dir / "finetune_search_checkpoints"
ft_search_csv_path = results_dir / "finetune_lr_search_layer_d.csv"

for search_layer, (X_ft_train, X_ft_val, y_ft_train, y_ft_val) in ft_search_samples.items():
    for lr in learning_rates:
        print(f"\n=== {search_layer} | learning_rate={lr} ===")
        t0 = time.time()

        try:
            history_logger = HistoryLogger()

            clf = FinetunedTabPFNClassifier(
                device="cuda",
                epochs=max_epochs,
                learning_rate=lr,
                early_stopping=True,
                early_stopping_patience=patience,
                eval_metric="roc_auc",
                experiment_logger=history_logger,
                n_estimators_finetune=tabpfn_n_estimators,
                n_inference_subsample_samples=tabpfn_subsample_samples,
                n_finetune_ctx_plus_query_samples=10_000,
            )
            clf.fit(
                X_ft_train, y_ft_train,
                X_val=X_ft_val, y_val=y_ft_val,
                output_dir=ft_checkpoint_dir / search_layer / f"lr_{lr}",
            )
            total_time = time.time() - t0

            val_history = [r for r in history_logger.epoch_records if val_metric_key in r]
            best_record = max(val_history, key=lambda r: r[val_metric_key])
            result = {
                "layer": search_layer,
                "learning_rate": lr,
                "best_epoch": int(best_record["train/epoch"]),
                "best_val_auc": best_record[val_metric_key],
                "epochs_run": len(val_history),
                "total_search_seconds": total_time,
                "status": "ok",
            }

        except Exception as e:
            # OOM or any other failure on this LR: log it and move on instead of
            # taking down the whole search.
            print(f"  FAILED: {type(e).__name__}: {e}")
            result = {
                "layer": search_layer,
                "learning_rate": lr,
                "best_epoch": None,
                "best_val_auc": None,
                "epochs_run": None,
                "total_search_seconds": time.time() - t0,
                "status": f"failed: {type(e).__name__}",
            }

        finally:
            # Save after every combo, success or failure, so a crash on combo N
            # still leaves combos 1..N-1 on disk instead of losing the whole run.
            ft_search_results.append(result)
            pd.DataFrame(ft_search_results).to_csv(ft_search_csv_path, index=False)

            for name in ("clf", "history_logger"):
                if name in dir():
                    del locals()[name]
            gc.collect()
            torch.cuda.empty_cache()

ft_search_df_results = pd.DataFrame(ft_search_results)
ft_search_df_results

/usr/local/lib/python3.12/dist-packages/tabpfn/finetuning/finetuned_classifier.py:175: UserWarning: `use_fixed_preprocessing_seed` should only be used if `n_estimators_finetune` == `n_estimators_validation` == `n_estimators_final_inference`. Consider setting the number of estimators for validation and final inference to the same value as `n_estimators_finetune`(=2).
  super().__init__(
/usr/local/lib/python3.12/dist-packages/tabpfn/finetuning/finetuned_base.py:877: UserWarning: Output dir present but no checkpoint file found for train_size=85000. Starting from default checkpoint and epoch 0
  get_checkpoint_path_and_epoch_from_output_dir(



=== layer_d_bts_meteostat_gdelt_subsampled | learning_rate=3e-05 ===


tabpfn-v3.5-20260909.safetensors:   0%|          | 0.00/876M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

Finetuning Epoch 1/30:   0%|          | 0/9 [00:00<?, ?it/s]

...

Finetuning Epoch 7/30:   0%|          | 0/9 [00:00<?, ?it/s]

    epoch 6: val_auc=0.6982


Finetuning Epoch 8/30:   0%|          | 0/9 [00:00<?, ?it/s]

    epoch 7: val_auc=0.6982


,layer,learning_rate,best_epoch,best_val_auc,epochs_run,total_search_seconds,status
0,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-05,4,0.698998,13,1931.558442,ok
1,layer_d_bts_meteostat_gdelt_subsampled,1.000000e-05,7,0.699618,16,2357.233851,ok
2,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-06,0,0.698462,8,1183.653193,ok
3,layer_d_bts_meteostat_gdelt_subsampled,1.000000e-06,0,0.698523,8,1184.660510,ok
4,layer_d_bts_meteostat_gdelt_subsampled,3.000000e-07,0,0.698542,8,1184.411711,ok
